# Invocations API: Concurrent Pr Review

This notebook mirrors `review_bot.scenarios.concurrent_pr_review`. It teaches the custom invocation payload, scenario metadata, pattern anatomy, agent roles, workflow execution, and the matching Invocations API response contract.

## 1. Notebook Setup

Run this notebook from the project virtual environment after installing the package with `python -m pip install -e . --no-deps`.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start):
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "review_bot").exists():
            return candidate
        nested = candidate / "invocations-api-review-bot"
        if (nested / "src" / "review_bot").exists():
            return nested
    raise RuntimeError("Could not find invocations-api-review-bot project root.")

PROJECT_ROOT = find_project_root(Path.cwd())
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

PROJECT_ROOT

## 2. Load The Scenario And Helpers

Each scenario has its own Python module with a single `SCENARIO` object. Notebook helpers keep repeated learning/display code out of the notebook cells.

In [ ]:
from review_bot.scenarios.concurrent_pr_review import SCENARIO
from review_bot.scenarios import SCENARIOS_BY_ID, get_scenario
from review_bot.notebook_helpers import (
    agent_roster,
    default_ollama_options,
    invocation_reference,
    load_sample_payload,
    pattern_anatomy,
    scenario_summary,
)

scenario = SCENARIO
assert SCENARIOS_BY_ID["concurrent-pr-review"] is scenario
assert get_scenario("concurrent-pr-review") is scenario

scenario_summary(scenario)

## 3. Pattern Anatomy

This is the orchestration behavior. The Invocations API boundary owns the request and response JSON, while the payload selects the scenario.

In [ ]:
pattern_anatomy(scenario)

## 4. Agent Roster

The roster shows who participates and what each agent is instructed to do.

In [ ]:
agent_roster(scenario)

## 5. Load And Validate The Invocation Payload

Invocations owns its JSON contract. This is the same sample payload you can POST to `/invocations`.

In [ ]:
from review_bot.models import build_review_prompt, parse_review_request

payload = load_sample_payload(PROJECT_ROOT, scenario)
request = parse_review_request(payload)
assert request.scenario == scenario.id
request

## 6. Inspect The Prompt Built From The Payload

The custom invocation fields become a prompt for the selected workflow. This is the key contrast with the Responses API notebooks.

In [ ]:
prompt = build_review_prompt(request)
print(prompt)

## 7. Live In-Process Run

This cell calls Ollama through the same `run_review` function used by the Invocations handler.

In [ ]:
from review_bot.workflows import run_review

ollama_options = default_ollama_options()
response = await run_review(request, **ollama_options)
response.to_dict()

## 8. Read The Structured Result

The Invocations API returns an application-owned response contract with metadata, summary, recommendations, and session information.

In [ ]:
print(f"Scenario: {response.scenario}")
print(f"Pattern: {response.pattern}")
print(f"Agents: {', '.join(response.agents)}")
print("
Summary:
")
print(response.summary[:5000])
print("
Recommendations:
")
for item in response.recommendations:
    print(f"- {item}")

## 9. Invocations API Boundary

Use this reference when you want to compare the in-process workflow with the hosted `/invocations` endpoint.

In [ ]:
invocation_reference(scenario, request)

## 10. Experiments

Try changing `constraints`, `artifacts`, `max_tokens`, `temperature`, or one agent instruction in the scenario module. Re-run prompt construction and the live call to compare results.